# Vision Models Pipeline

End-to-end comparison of three vision-language models on local images:

| Model | Task | Source |
|-------|------|--------|
| **ViT-Base** (`google/vit-base-patch16-224`) | Image classification (1000 ImageNet classes) | HuggingFace |
| **BLIP-base** (`Salesforce/blip-image-captioning-base`) | Free-form image captioning | HuggingFace |
| **CLIP** (`openai/clip-vit-base-patch32`) | Semantic image search via cosine similarity | HuggingFace |

## Setup
Install dependencies before opening this notebook:
```bash
uv pip install -e .
```
Models are downloaded from HuggingFace on first run (~346 MB for ViT, ~990 MB for BLIP, ~350 MB for CLIP)
and cached in `~/.cache/huggingface/hub`. They are reused on subsequent runs without re-downloading.

## Sample images
Place `.jpg` / `.png` / `.jpeg` files in the `images/` folder next to this notebook.
Five sample images are already included.

In [ ]:
# =============================================================================
# QUICK INSTALLATION CHECK
# =============================================================================

def check_gpu_setup():
    """Verify PyTorch installation and available accelerators."""
    print("CHECKING INSTALLATION...")
    print("-" * 50)
    try:
        import torch
        print(f"PyTorch: v{torch.__version__}")
        accelerators = []
        if torch.cuda.is_available():
            accelerators.append(f"CUDA: {torch.cuda.get_device_name(0)}")
        if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            accelerators.append("MPS: Apple Silicon GPU")
        if accelerators:
            for acc in accelerators:
                print(f"   {acc}")
        else:
            print("   No GPU available — using CPU")
    except ImportError:
        print("PyTorch not installed")
    except Exception as e:
        print(f"PyTorch error: {e}")

check_gpu_setup()

In [ ]:
import torch
from pathlib import Path
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    BlipProcessor, BlipForConditionalGeneration,
    CLIPProcessor, CLIPModel,
)
from PIL import Image
import requests
import matplotlib.pyplot as plt
from io import BytesIO

In [ ]:
def setup_device():
    """Select the best available compute device: CUDA > MPS > CPU."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"NVIDIA GPU: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Apple Silicon GPU (MPS)")
    else:
        device = torch.device("cpu")
        print("CPU (no GPU acceleration)")
    return device

device = setup_device()

In [ ]:
# =============================================================================
# IMAGE FOLDER SETUP AND UTILITIES
# =============================================================================

# Relative to this notebook's working directory — works regardless of where the
# repo is cloned, as long as the notebook is opened from the repo root.
IMAGES_FOLDER = Path("./images")

def setup_images_folder():
    """Create the images folder if it doesn't exist."""
    IMAGES_FOLDER.mkdir(parents=True, exist_ok=True)
    print(f"Images folder: {IMAGES_FOLDER.resolve()}")

def load_image_from_file(filename):
    """
    Load an image from the images folder.
    Raises FileNotFoundError immediately if the file is missing,
    so failures are obvious at the source rather than several steps later.
    """
    path = IMAGES_FOLDER / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Image not found: {path.resolve()}\n"
            f"Place the file in the '{IMAGES_FOLDER}/' folder."
        )
    image = Image.open(path).convert('RGB')
    print(f"Loaded: {filename}")
    return image

def load_image_from_url(url, timeout=10):
    """Load an image from a URL."""
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert('RGB')

def list_available_images():
    """List all image files in IMAGES_FOLDER."""
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.gif', '*.tiff']
    files = []
    for ext in extensions:
        files.extend(IMAGES_FOLDER.glob(ext))
        files.extend(IMAGES_FOLDER.glob(ext.upper()))
    names = sorted({f.name for f in files})
    if not names:
        print(f"No images found in {IMAGES_FOLDER.resolve()}")
        print("Add .jpg / .png files to the 'images/' folder.")
    else:
        print(f"Available images ({len(names)}):")
        for i, name in enumerate(names, 1):
            print(f"   {i}. {name}")
    return names

def show_image(image, title="Image", figsize=(8, 6)):
    """Display an image with matplotlib."""
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()

def show_available_images():
    """Display a 2x2 preview grid of the first four available images."""
    names = list_available_images()
    if not names:
        return []

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    for i, name in enumerate(names[:4]):
        try:
            img = load_image_from_file(name)
            axes[i].imshow(img)
            axes[i].set_title(name, fontsize=10)
        except FileNotFoundError as e:
            axes[i].text(0.5, 0.5, 'Load error', ha='center', va='center')
            axes[i].set_title(name, fontsize=10)
        axes[i].axis('off')
    for i in range(len(names), 4):
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
    return names

setup_images_folder()

In [ ]:
# =============================================================================
# IMAGE CLASSIFIER  (Vision Transformer)
# =============================================================================

class ImageClassifier:
    """
    Image classifier based on Vision Transformer (ViT).

    Uses Google's ViT-Base pre-trained on ImageNet-1K (1000 classes).
    Architecture: transformer with 16x16 pixel patches, 224x224 input resolution.
    """

    def __init__(self, device):
        self.device = device
        print("Loading Vision Transformer...")
        model_name = "google/vit-base-patch16-224"
        # Preprocessor resizes input to 224x224
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        # Pre-trained weights from ImageNet-1K
        self.model = AutoModelForImageClassification.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        print(f"Model loaded: {model_name}")
        print(f"   Device: {device}")
        print(f"   Classes: {len(self.model.config.id2label)}")
        # Note: first inference is slower on MPS (Metal shader compilation on demand).
        # Subsequent calls on the same model will be significantly faster.

    def classify(self, image, top_k=5):
        """
        Classify an image and return the top-k predictions.

        Pipeline:
        1. Preprocess: resize, normalize, tokenize into patches
        2. Forward pass: feature extraction via transformer blocks
        3. Classification: linear projection to 1000 classes
        4. Post-process: softmax + probability ranking
        """
        if not 1 <= top_k <= 1000:
            raise ValueError(f"top_k must be between 1 and 1000, got: {top_k}")

        inputs = self.processor(image, return_tensors="pt").to(self.device)

        # inference_mode is stricter than no_grad and slightly faster
        with torch.inference_mode():
            outputs = self.model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits[0], dim=-1)

        top = torch.topk(probs, k=min(top_k, probs.size(0)))
        results = [
            (self.model.config.id2label[top.indices[i].item()], top.values[i].item())
            for i in range(len(top.values))
        ]
        return results

classifier = ImageClassifier(device)

In [ ]:
available_images = list_available_images()
if available_images:
    test_image_name = available_images[0]
    test_image = load_image_from_file(test_image_name)
    show_image(test_image, f"Classifying: {test_image_name}")

    print("\nClassifying...")
    results = classifier.classify(test_image, top_k=5)

    print("\nTop 5 predictions:")
    print("-" * 50)
    for i, (label, prob) in enumerate(results, 1):
        print(f"{i}. {label:<30} {prob:.1%}")
else:
    print("No images available — add files to the 'images/' folder.")

In [ ]:
# =============================================================================
# IMAGE CAPTIONER  (BLIP)
# =============================================================================

class ImageCaptioner:
    """
    Image description generator based on BLIP
    (Bootstrapping Language-Image Pre-training).

    Uses Salesforce BLIP-base, pre-trained on 14M image-text pairs.
    Architecture: encoder-decoder with ViT for vision and BERT for language.
    Input resolution: 384x384.
    """

    def __init__(self, device):
        self.device = device
        print("Loading BLIP...")
        model_name = "Salesforce/blip-image-captioning-base"
        # Multimodal preprocessor: resizes image to 384x384, tokenizes text with BERT WordPiece
        self.processor = BlipProcessor.from_pretrained(model_name)
        # Encoder-decoder: ViT extracts visual features, BERT decoder generates tokens
        self.model = BlipForConditionalGeneration.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        print(f"Model loaded: {model_name}")
        print(f"   Device: {device}")
        print(f"   Input resolution: 384x384")
        # Note: first inference is slower on MPS (Metal shader compilation on demand).
        # Subsequent calls on the same model will be significantly faster.

    def generate_caption(self, image, max_length=50):
        """
        Generate an automatic caption for the image using beam search.

        Pipeline:
        1. Visual encoding: ViT processes image -> features [197, 768]
        2. Autoregressive decoding: generates tokens sequentially
        3. Beam search: explores multiple hypotheses in parallel (num_beams=5)
        4. Stopping: on [SEP] token or max_length
        """
        inputs = self.processor(image, return_tensors="pt").to(self.device)
        # inference_mode is stricter than no_grad and slightly faster
        with torch.inference_mode():
            generated_ids = self.model.generate(
                **inputs,
                max_length=max_length,
                num_beams=5,
                early_stopping=True,
            )
        # BLIP generates clean lowercase text by design; no post-processing needed
        return self.processor.decode(generated_ids[0], skip_special_tokens=True)

captioner = ImageCaptioner(device)

In [ ]:
if available_images:
    test_image_name = available_images[0]
    test_image = load_image_from_file(test_image_name)
    show_image(test_image, f"Captioning: {test_image_name}")

    print("\nGenerating caption...")
    caption = captioner.generate_caption(test_image)
    print(f"Caption: {caption}")
else:
    print("No images available — add files to the 'images/' folder.")

In [ ]:
# =============================================================================
# SEMANTIC IMAGE SEARCHER  (CLIP)
# =============================================================================

class SemanticImageSearcher:
    """
    Text-to-image semantic search using OpenAI CLIP.

    Encodes both images and text into a shared embedding space,
    then ranks images by cosine similarity to a query string.
    """

    def __init__(self, device):
        self.device = device
        print("Loading CLIP...")
        model_name = "openai/clip-vit-base-patch32"
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model = CLIPModel.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        print(f"Model loaded: {model_name}")
        # Note: first inference is slower on MPS (Metal shader compilation on demand).
        # Subsequent calls on the same model will be significantly faster.

    def encode_images(self, images):
        """Encode a list of images to L2-normalized CLIP embeddings."""
        inputs = self.processor(images=images, return_tensors="pt").to(self.device)
        with torch.inference_mode():
            # transformers 5.x: get_image_features() returns BaseModelOutputWithPooling,
            # not a plain tensor. Go through vision_model + projection explicitly.
            vision_out = self.model.vision_model(**inputs)
            embeddings = self.model.visual_projection(vision_out.pooler_output)
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        return embeddings

    def encode_text(self, texts):
        """Encode a list of text strings to L2-normalized CLIP embeddings."""
        inputs = self.processor(text=texts, return_tensors="pt", padding=True).to(self.device)
        with torch.inference_mode():
            # Same workaround as encode_images — use text_model + projection directly.
            text_out = self.model.text_model(**inputs)
            embeddings = self.model.text_projection(text_out.pooler_output)
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        return embeddings

    def similarity_search(self, images, query_texts, top_k=None):
        """
        Return (index, cosine_similarity) pairs for all images,
        ranked by similarity to each query in query_texts.

        query_texts can be a single string or a list of strings.
        When a list is passed, results are averaged across all queries
        before ranking — useful for concept scoring in analyze_image().
        """
        if isinstance(query_texts, str):
            query_texts = [query_texts]

        image_embeddings = self.encode_images(images)
        text_embeddings = self.encode_text(query_texts)

        # Average similarity across all query texts before ranking
        similarities = torch.matmul(text_embeddings, image_embeddings.t()).mean(dim=0)

        if similarities.dim() == 0:
            similarities = similarities.unsqueeze(0)

        if top_k is None:
            top_k = len(images)

        if len(images) == 1:
            return [(0, similarities[0].item())]

        top_indices = torch.topk(similarities, k=min(top_k, len(images))).indices
        return [(i.item(), similarities[i].item()) for i in top_indices]

searcher = SemanticImageSearcher(device)

In [ ]:
print("Loading images for semantic search...")

if len(available_images) >= 2:
    search_images = []
    search_names = []
    for name in available_images[:6]:
        img = load_image_from_file(name)
        search_images.append(img)
        search_names.append(name)
    print(f"{len(search_images)} images ready for search")

    queries = [
        "an animal",
        "a person",
        "nature and landscapes",
        "buildings and architecture",
        "food",
        "vehicle or transport",
    ]

    print("\nSEMANTIC SEARCH RESULTS")
    print("=" * 50)
    for query in queries:
        print(f"\nQuery: '{query}'")
        print("-" * 30)
        results = searcher.similarity_search(search_images, query, top_k=3)
        for rank, (idx, score) in enumerate(results, 1):
            if idx < len(search_names):
                print(f"{rank}. {search_names[idx]:<20} (similarity: {score:.3f})")
else:
    print("Need at least 2 images for semantic search.")
    print("Add more images to the 'images/' folder.")

In [ ]:
# =============================================================================
# FULL PIPELINE: classify + caption + semantic similarity
# =============================================================================

def analyze_image(image_name, description=None):
    """Run all three models on a single local image."""
    if description is None:
        description = image_name

    print(f"\nFULL ANALYSIS: {description}")
    print("=" * 60)

    image = load_image_from_file(image_name)
    show_image(image, f"Analysis: {description}")

    print("1. CLASSIFICATION (Top 3):")
    print("-" * 25)
    classifications = classifier.classify(image, top_k=3)
    for i, (label, prob) in enumerate(classifications, 1):
        print(f"   {i}. {label} ({prob:.1%})")

    print("\n2. AUTOMATIC CAPTION:")
    print("-" * 30)
    caption = captioner.generate_caption(image)
    print(f"   {caption}")

    print("\n3. CONCEPT SIMILARITY:")
    print("-" * 35)
    test_concepts = ["animal", "person", "nature", "building", "food", "vehicle"]
    # Encode all concepts in a single CLIP forward pass, then score each against the image
    image_embedding = searcher.encode_images([image])
    text_embeddings = searcher.encode_text(test_concepts)
    scores = torch.matmul(text_embeddings, image_embedding.t()).squeeze()
    for concept, score in zip(test_concepts, scores.tolist()):
        bar = "█" * int(score * 20)
        print(f"   {concept:<10} {bar:<20} {score:.3f}")


# Re-query the folder each time so newly added images are visible without
# re-running earlier cells.
available_images = list_available_images()

# Change image_index to analyze a different file
image_index = 0
if available_images:
    analyze_image(available_images[image_index])
else:
    print("No images available — add files to the 'images/' folder.")